In [1]:
import urllib.request
import zipfile
import os
from pathlib import Path

In [2]:
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path)/"SMSSpamCollection.tsv"

In [3]:
def download_and_unzip_spam_data(url,zip_path,extracted_path,data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return
    with urllib.request.urlopen(url) as response:
        with open(zip_path,"wb") as out_file:
            out_file.write(response.read())
    with zipfile.ZipFile(zip_path,"r") as zip_ref:
        zip_ref.extractall(extracted_path)
    original_file_path = Path(extracted_path)/"SMSSpamCollection"
    os.rename(original_file_path,data_file_path)
    print(f"File download and saved as {data_file_path}")
download_and_unzip_spam_data(url,zip_path,extracted_path,data_file_path)

File download and saved as sms_spam_collection/SMSSpamCollection.tsv


In [4]:
import pandas as pd
df = pd.read_csv(data_file_path,sep='\t',header=None,names=["Label","Text"])
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [5]:
print(df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


In [6]:
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    ham_subset = df[df["Label"] == "ham"].sample(num_spam,random_state=123)
    balanced_df = pd.concat([ham_subset,df[df["Label"] == "spam"]])
    return balanced_df
balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [7]:
balanced_df["Label"] = balanced_df["Label"].map({"ham":0,"spam":1})

In [8]:
def random_split(df,train_frac,validation_frac):
    df = df.sample(frac=1,random_state=123).reset_index(drop=True)
    train_end = int(len(df)*train_frac)
    validation_end = train_end+int(len(df)*validation_frac)
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]
    return train_df,validation_df,test_df
train_df,validation_df,test_df = random_split(balanced_df,0.7,0.1)

In [9]:
train_df.to_csv("train.csv",index=None)
validation_df.to_csv("validation.csv",index=None)
test_df.to_csv("test.csv",index=None)

In [11]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>",allowed_special={"<|endoftext|>"}))

[50256]


In [18]:
import torch
from torch.utils.data import Dataset

class SpamDataset(Dataset):
    def __init__(self,csv_file,tokenizer,max_length=None,pad_token_id=50256):
        self.data = pd.read_csv(csv_file)
        self.encoded_texts = [tokenizer.encode(text) for text in self.data["Text"]]
        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            self.encoded_texts = [encoded_text[:self.max_length] for encoded_text in self.encoded_texts]
        self.encoded_texts = [encoded_text+[pad_token_id]*(self.max_length-len(encoded_text)) for encoded_text in self.encoded_texts]
    def __getitem__(self,index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (torch.tensor(encoded,dtype=torch.long),torch.tensor(label,dtype=torch.long))
    def __len__(self):
        return len(self.data)
    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length

In [19]:
train_dataset = SpamDataset(csv_file="train.csv",max_length=None,tokenizer=tokenizer)

In [20]:
print(train_dataset.max_length)

120


In [21]:
val_dataset = SpamDataset(csv_file="validation.csv",max_length=train_dataset.max_length,tokenizer=tokenizer)
test_dataset = SpamDataset(csv_file="test.csv",max_length=train_dataset.max_length,tokenizer=tokenizer)

In [22]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8
torch.manual_seed(123)

train_loader = DataLoader(
    dataset = train_dataset,
    batch_size = batch_size,
    shuffle = True,
    num_workers=num_workers,
    drop_last=True
)
val_loader = DataLoader(
    dataset = val_dataset,
    batch_size = batch_size,
    shuffle = True,
    num_workers = num_workers,
    drop_last = True
)
test_loader = DataLoader(
    dataset = test_dataset,
    batch_size = batch_size,
    shuffle = True,
    num_workers = num_workers,
    drop_last = True
)


In [23]:
for input_batch,target_batch in train_loader:
    pass
print("Input batch dimensions:",input_batch.shape)
print("Label batch dimensions:",target_batch.shape)

Input batch dimensions: torch.Size([8, 120])
Label batch dimensions: torch.Size([8])


In [24]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

130 training batches
18 validation batches
37 test batches


In [25]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"
BASE_CONFIG = {
    "vocab_size":50257,
    "context_length":1024,
    "drop_rate":0.0,
    "qkv_bias":True
}
model_configs = {
    "gpt2-small (124M)":{"emb_dim":768,"n_layers":12,"n_heads":12},
    "gpt2-medium (355M)":{"emb_dim":1024,"n_layers":24,"n_heads":16},
    "gpt2-large (774M)":{"emb_dim":1280,"n_layers":36,"n_heads":20},
    "gpt2-x1 (1558M)":{"emb_dim":1600,"n_layers":48,"n_heads":25},
}
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [29]:
from gpt2_hf_download import download_and_load_gpt2_hf
from chapter05 import GPTModel,load_weights_into_gpt

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings,params = download_and_load_gpt2_hf(
    model_size=model_size,models_dir="gpt2-hf"
)

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model,params)
model.eval()

📡 下载源: ModelScope (modelscope.cn)
📥 下载 GPT-2 124M 到 gpt2-hf/124M...
✅ 模型文件已存在，跳过下载
🔄 转换 PyTorch 权重为书籍格式...
  Key prefix: '' (detected from 'wte.weight')
✅ GPT-2 124M 加载完成！


GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7

In [30]:
from chapter05 import generate_text_simple,text_to_token_ids,token_ids_to_text

text_1 = "Every effort moves you"
token_ids = generate_text_simple(
    model = model,
    idx = text_to_token_ids(text_1,tokenizer),
    max_new_tokens=15,
    context_size = BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids,tokenizer))

Every effort moves you forward.

The first step is to understand the importance of your work


In [31]:
text_2 = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':"
    " 'You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award.'"
)
token_ids = generate_text_simple(
    model = model,
    idx = text_to_token_ids(text_2,tokenizer),
    max_new_tokens=23,
    context_size = BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids,tokenizer))

Is the following text 'spam'? Answer with 'yes' or 'no': 'You are a winner you have been specially selected to receive $1000 cash or a $2000 award.'

The following text 'spam'? Answer with 'yes' or 'no': 'You are a winner


In [32]:
print(model)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7

In [33]:
for param in model.parameters():
    param.required_grad = False

In [35]:
torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(
    in_features = BASE_CONFIG["emb_dim"],
    out_features = num_classes
)

In [36]:
for param in model.trf_blocks[-1].parameters():
    param.required_grad = True
for param in model.final_norm.parameters():
    param.required_grad = True

In [37]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print("Input:",inputs)
print("Inputs dimensions:",inputs.shape)

Input: tensor([[5211,  345,  423,  640]])
Inputs dimensions: torch.Size([1, 4])


In [38]:
with torch.no_grad():
    outputs = model(inputs)
print("Outputs:\n",outputs)
print("Outputs dimension:",outputs.shape)

Outputs:
 tensor([[[-1.5854,  0.9904],
         [-3.7235,  7.4548],
         [-2.2661,  6.6049],
         [-3.5983,  3.9902]]])
Outputs dimension: torch.Size([1, 4, 2])


In [39]:
print("Last output token:",outputs[:,-1,:])

Last output token: tensor([[-3.5983,  3.9902]])


In [40]:
print("Last output token:",outputs[:,-1,:])

Last output token: tensor([[-3.5983,  3.9902]])


In [41]:
probas = torch.softmax(outputs[:,-1,:],dim=-1)
label = torch.argmax(probas)
print("Class label:",label.item())

Class label: 1


In [42]:
logits = outputs[:,-1,:]
label = torch.argmax(logits)
print("Class label:",label.item())

Class label: 1


In [43]:
def calc_accuracy_loader(data_loader,model,device,num_batches=None):
    model.eval()
    correct_predictions,num_examples=0,0
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches,len(data_loader))
    for i,(input_batch,target_batch) in enumerate(data_loader):
        if i<num_batches:
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)
            with torch.no_grad():
                logits = model(input_batch)[:,-1,:]
            predicted_labels = torch.argmax(logits,dim=-1)
            num_examples += predicted_labels.shape[0]
            correct_predictions += (
                (predicted_labels == target_batch).sum().item()
            )
        else:
            break
    return correct_predictions/num_examples

In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
torch.manual_seed(123)
train_accuracy = calc_accuracy_loader(
    train_loader,model,device,num_batches=10
)
val_accuracy = calc_accuracy_loader(
    val_loader,model,device,num_batches=10
)
test_accuracy = calc_accuracy_loader(
    test_loader,model,device,num_batches=10
)

In [47]:
print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Training accuracy: 46.25%
Validation accuracy: 53.75%
Test accuracy: 50.00%


In [48]:
def calc_loss_batch(input_batch,target_batch,model,device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)[:,-1,:]
    loss = torch.nn.functional.cross_entropy(logits,target_batch)
    return loss

In [49]:
def calc_loss_loader(data_loader,model,device,num_batches=None):
    total_loss = 0.
    if len(data_loader)==0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches,len(data_loader))
    for i,(input_batch,target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(
                input_batch,target_batch,model,device
            )
            total_loss += loss.item()
        else:
            break
    return total_loss/num_batches

In [50]:
with torch.no_grad():
    train_loss = calc_loss_loader(
        train_loader,model,device,num_batches=5
    )
    val_loss = calc_loss_loader(val_loader,model,device,num_batches=5)
    test_loss = calc_loss_loader(test_loader,model,device,num_batches=5)
print(f"Trainging loss:{train_loss:.3f}")
print(f"Validation loss:{val_loss:.3f}")
print(f"Test loss:{test_loss:.3f}")

Trainging loss:3.211
Validation loss:2.436
Test loss:2.552


In [57]:
def train_classifier_simple(
    model,train_loader,val_loader,optimizer,device,num_epochs,eval_freq,eval_iter):
    train_losses,val_losses,train_accs,val_accs = [],[],[],[]
    examples_seen,global_step = 0,-1
    for epoch in range(num_epochs):
        model.train()
        for input_batch,target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(
                input_batch,target_batch,model,device
            )
            loss.backward()
            optimizer.step()
            examples_seen += input_batch.shape[0]
            global_step += 1
            if global_step % eval_freq == 0:
                train_loss,val_loss = evaluate_model(
                    model,train_loader,val_loader,device,eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}):"
                      f"Train loss {train_loss:.3f},"
                      f"Val loss {val_loss:.3f}"
                )
        train_accuracy = calc_accuracy_loader(
            train_loader,model,device,num_batches=eval_iter
        )
        val_accuracy = calc_accuracy_loader(
            val_loader,model,device,num_batches=eval_iter
        )
        print(f"Training accuracy: {train_accuracy*100:.2f}% | ",end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)
    return train_losses,val_losses,train_accs,val_accs,examples_seen
        

In [58]:
def evaluate_model(model,train_loader,val_loader,device,eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(
            train_loader,model,device,num_batches=eval_iter
        )
        val_loss = calc_loss_loader(
            val_loader,model,device,num_batches=eval_iter
        )
    model.train()
    return train_loss,val_loss

In [59]:
import time

start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(),lr=5e-5,weight_decay=0.1)
num_epochs=5

train_losses,val_losses,train_accs,val_accs,examples_seen = \
    train_classifier_simple(
        model,train_loader,val_loader,optimizer,device,
        num_epochs=num_epochs,eval_freq=50,
        eval_iter=5
    )
end_time = time.time()
execution_time_minutes = (end_time-start_time)/60
print(f"Trainging completed in {execution_time_minutes:.2f} minutes.")

Ep 1 (Step 000000):Train loss 0.875,Val loss 1.160
Ep 1 (Step 000050):Train loss 0.223,Val loss 0.190
Ep 1 (Step 000100):Train loss 0.098,Val loss 0.256
Training accuracy: 92.50% | Validation accuracy: 95.00%
Ep 2 (Step 000150):Train loss 0.105,Val loss 0.088
Ep 2 (Step 000200):Train loss 0.144,Val loss 0.264
Ep 2 (Step 000250):Train loss 0.036,Val loss 0.094
Training accuracy: 97.50% | Validation accuracy: 100.00%
Ep 3 (Step 000300):Train loss 0.002,Val loss 0.006
Ep 3 (Step 000350):Train loss 0.016,Val loss 0.015
Training accuracy: 100.00% | Validation accuracy: 97.50%
Ep 4 (Step 000400):Train loss 0.002,Val loss 0.001
Ep 4 (Step 000450):Train loss 0.000,Val loss 0.089
Ep 4 (Step 000500):Train loss 0.034,Val loss 0.200
Training accuracy: 100.00% | Validation accuracy: 97.50%
Ep 5 (Step 000550):Train loss 0.000,Val loss 0.085
Ep 5 (Step 000600):Train loss 0.000,Val loss 0.007
Training accuracy: 100.00% | Validation accuracy: 97.50%
Trainging completed in 1.51 minutes.


In [60]:
train_accuracy = calc_accuracy_loader(train_loader,model,device)
val_accuracy = calc_accuracy_loader(val_loader,model,device)
test_accuracy = calc_accuracy_loader(test_loader,model,device)
print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

Training accuracy: 100.00%
Validation accuracy: 97.22%
Test accuracy: 97.30%


In [61]:
def classify_review(text,model,tokenizer,device,max_length=None,pad_token_id=50256):
    model.eval()
    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[1]
    input_ids = input_ids[:min(max_length,supported_context_length)]
    input_ids += [pad_token_id]*(max_length-len(input_ids))
    input_tensor = torch.tensor(input_ids,device=device).unsqueeze(0)
    with torch.no_grad():
        logits = model(input_tensor)[:,-1,:]
    predicted_label = torch.argmax(logits,dim=-1).item()
    return "spam" if predicted_label == 1 else "not spam"

In [63]:
text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)
print(classify_review(text_1,model,tokenizer,device,max_length=train_dataset.max_length))
text_2 = (
    "Hey, just wanted to check if we're still on"
    " for dinner tonight? Let me know!"
)
print(classify_review(text_2,model,tokenizer,device,max_length=train_dataset.max_length))

torch.save(model.state_dict(),"review_classifier.pth")
model_state_dict = torch.load("review_classifier.pth",map_location=device)
model.load_state_dict(model_state_dict)

spam
not spam


<All keys matched successfully>